In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Connect to Data
conn = sqlite3.connect('data.sqlite')

In [ ]:
query_ca_promo = """
SELECT customerName, state, creditLimit
FROM customers
WHERE state = 'CA' AND creditLimit > 25000;
"""
ca_promo_df = pd.read_sql(query_ca_promo, conn)
ca_promo_df

In [ ]:
query_intl_collect = """
SELECT customerName, country
FROM customers
WHERE country != 'USA' AND customerName LIKE '%Collect%';
"""
intl_collect_df = pd.read_sql(query_intl_collect, conn)
intl_collect_df

In [ ]:
query_usa_credit = """
SELECT state, AVG(creditLimit) AS avg_credit_limit
FROM customers
WHERE country = 'USA'
GROUP BY state
ORDER BY avg_credit_limit DESC;
"""
usa_credit_df = pd.read_sql(query_usa_credit, conn)

plt.figure(figsize=(10, 6))
sns.barplot(data=usa_credit_df, x='state', y='avg_credit_limit', palette='viridis')
plt.title('Average Credit Limit by State (USA)')
plt.xlabel('State')
plt.ylabel('Average Credit Limit ($)')
plt.show()

In [ ]:
query_top_customers = """
SELECT c.customerName, SUM(p.amount) AS total_payments
FROM customers c
JOIN payments p ON c.customerNumber = p.customerNumber
GROUP BY c.customerNumber
ORDER BY total_payments DESC
LIMIT 10;
"""
top_customers_df = pd.read_sql(query_top_customers, conn)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_customers_df, x='total_payments', y='customerName', palette='mako')
plt.title('Top 10 Customers by Total Payments')
plt.xlabel('Total Payments ($)')
plt.ylabel('Customer Name')
plt.show()

In [ ]:
query_top_quantities = """
SELECT c.customerName, p.productName, SUM(od.quantityOrdered) AS total_quantity
FROM customers c
JOIN orders o ON c.customerNumber = o.customerNumber
JOIN orderdetails od ON o.orderNumber = od.orderNumber
JOIN products p ON od.productCode = p.productCode
GROUP BY c.customerNumber, p.productCode
HAVING total_quantity >= 10
ORDER BY total_quantity ASC;
"""
top_quantities_df = pd.read_sql(query_top_quantities, conn)
top_quantities_df.head()

In [ ]:
query_prod_analysis = """
SELECT p.productLine, COUNT(DISTINCT p.productCode) AS number_of_products, SUM(od.quantityOrdered) AS total_quantity_ordered
FROM products p
JOIN orderdetails od ON p.productCode = od.productCode
GROUP BY p.productLine;
"""
prod_analysis_df = pd.read_sql(query_prod_analysis, conn)

plt.figure(figsize=(8, 5))
sns.scatterplot(data=prod_analysis_df, x='number_of_products', y='total_quantity_ordered', s=150, color='crimson')
for i in range(prod_analysis_df.shape[0]):
    plt.text(prod_analysis_df.number_of_products[i] + 0.5, prod_analysis_df.total_quantity_ordered[i], 
             prod_analysis_df.productLine[i], fontsize=9)
plt.title('Relationship Between Number of Products and Total Orders by Product Line')
plt.xlabel('Number of Unique Products')
plt.ylabel('Total Quantity Ordered')
plt.grid(True)
plt.show()

In [ ]:
query_remote = """
SELECT e.firstName, e.lastName, e.jobTitle, 
       m.firstName || ' ' || m.lastName AS supervisorName, 
       o.city
FROM employees e
LEFT JOIN employees m ON e.reportsTo = m.employeeNumber
JOIN offices o ON e.officeCode = o.officeCode
WHERE e.officeCode IN (
    SELECT officeCode
    FROM employees
    GROUP BY officeCode
    HAVING COUNT(employeeNumber) < 5
);
"""
remote_offices_df = pd.read_sql(query_remote, conn)
remote_offices_df

In [ ]:
# Step 9: Close the Connection
conn.close()

In [ ]:
import zipfile

zip_file_path = 'im.db.zip'
extract_to_path = './'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)

conn4 = sqlite3.connect('im.db')

schema_df = pd.read_sql("SELECT * FROM sqlite_master", conn4)
schema_df

In [ ]:
# Slide 3: Visual for Business Question
query_runtime = """
SELECT mb.runtime_minutes, mr.averagerating, mr.numvotes
FROM movie_basics mb
JOIN movie_ratings mr ON mb.movie_id = mr.movie_id
WHERE mb.start_year BETWEEN 2010 AND 2019 
  AND mr.numvotes > 10000 
  AND mb.runtime_minutes IS NOT NULL 
  AND mb.runtime_minutes < 240
"""
runtime_df = pd.read_sql(query_runtime, conn4)

plt.figure(figsize=(10, 6))
sns.histplot(x=runtime_df['runtime_minutes'], y=runtime_df['averagerating'], bins=30, pmax=0.8, cmap="mako")
plt.title('Movie Runtime vs Average Rating (2010-2019)')
plt.xlabel('Runtime (Minutes)')
plt.ylabel('Average Rating')
plt.show()